In [28]:
from functools import cache
from pathlib import Path

import pandas as pd
import google.auth
import geopandas as gpd

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.sql import get_engine

from shared_utils import bq_utils

from update_vars import GTFS_DATA_DICT, file_name

# Initialize credentials and DB engine
credentials, project = google.auth.default()
db_engine = get_engine()

import _prep_crosswalk_ntd
import _load_gtfs_data

In [2]:
@cache
def gcs_pandas():
    return GCSPandas()

In [3]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [21]:
crosswalk_df = bq_utils.download_table(
        project_name="cal-itp-data-infra",
        dataset_name="mart_transit_database",
        table_name="bridge_gtfs_analysis_name_x_ntd",
        date_col=None,
    )

Downloading: 100%|██████████|
query: SELECT * FROM  `cal-itp-data-infra`.`mart_transit_database`.`bridge_gtfs_analysis_name_x_ntd`


In [23]:
crosswalk_df.loc[crosswalk_df.analysis_name.str.contains("Banning")]

,organization_name,organization_source_record_id,schedule_source_record_id,schedule_gtfs_dataset_name,analysis_name,regional_feed_type,county_name,caltrans_district,caltrans_district_name,caltrans_district_full,ntd_id,ntd_id_2022,rtpa_name,mpo_name
117,City of Banning,recuGkFhN2WXGK67H,recnAiZYHWBxUwH0F,Banning Pass Schedule,City of Banning,None,Riverside,8,San Bernardino / Riverside,08 - San Bernardino / Riverside,None,None,Southern California Association of Governments,Southern California Association of Governments


In [24]:
crosswalk_df2 = (
        crosswalk_df
        .drop_duplicates(
            subset=["analysis_name", "organization_name", "schedule_gtfs_dataset_name"]
        )
        .reset_index()
    )


In [25]:
crosswalk_df2.loc[crosswalk_df2.analysis_name.str.contains("Banning")]

,index,organization_name,organization_source_record_id,schedule_source_record_id,schedule_gtfs_dataset_name,analysis_name,regional_feed_type,county_name,caltrans_district,caltrans_district_name,caltrans_district_full,ntd_id,ntd_id_2022,rtpa_name,mpo_name
117,117,City of Banning,recuGkFhN2WXGK67H,recnAiZYHWBxUwH0F,Banning Pass Schedule,City of Banning,None,Riverside,8,San Bernardino / Riverside,08 - San Bernardino / Riverside,None,None,Southern California Association of Governments,Southern California Association of Governments


In [26]:
crosswalk_df2 = crosswalk_df2.rename(columns={"schedule_gtfs_dataset_name": "name"})

crosswalk_df2["caltrans_district_int"] = crosswalk_df2.caltrans_district
crosswalk_df2.caltrans_district = crosswalk_df2.caltrans_district.apply(lambda x: '{0:0>2}'.format(x)) 
    
crosswalk_df2["caltrans_district"] = (
        crosswalk_df2.caltrans_district.astype(str) + "-" + crosswalk_df2.caltrans_district_name
    )

crosswalk_df2 = crosswalk_df2[
        [
            "name",
            "analysis_name",
            "county_name",
            "caltrans_district",
            "caltrans_district_int",
            "ntd_id",
            "ntd_id_2022",
        ]
    ]

In [27]:
crosswalk_df2.loc[crosswalk_df2.analysis_name.str.contains("Banning")]

,name,analysis_name,county_name,caltrans_district,caltrans_district_int,ntd_id,ntd_id_2022
117,Banning Pass Schedule,City of Banning,Riverside,08-San Bernardino / Riverside,8,None,None


In [29]:
full_function = _prep_crosswalk_ntd.load_crosswalk()

Downloading: 100%|██████████|
query: SELECT * FROM  `cal-itp-data-infra`.`mart_transit_database`.`bridge_gtfs_analysis_name_x_ntd`


In [31]:
full_function.loc[full_function.analysis_name.str.contains("Banning")]

,name,analysis_name,county_name,caltrans_district,caltrans_district_int,ntd_id,ntd_id_2022
117,Banning Pass Schedule,City of Banning,Riverside,08-San Bernardino / Riverside,8,None,None


In [33]:
june_crosswalk_df = f"{GTFS_DATA_DICT.gcs_paths.DIGEST_GCS}processed/{GTFS_DATA_DICT.gtfs_digest_rollup.crosswalk}_{file_name}.parquet"

In [34]:
crosswalk_df = gcs_pandas().read_parquet(crosswalk_url)[["name", "analysis_name"]].drop_duplicates()

In [36]:
crosswalk_df.loc[crosswalk_df.analysis_name.str.contains("Banning")]

,name,analysis_name
117,Banning Pass Schedule,City of Banning


## No Data Shown in Portfolio in `_load_gtfs_data`

In [11]:
from update_vars import GTFS_DATA_DICT, analysis_month, file_name, last_year, previous_month

In [37]:
# crosswalk_df

In [12]:
PROD_PROJECT = "cal-itp-data-infra"
PROD_MART = "mart_gtfs_rollup"
MONTH_DATE_COL = "month_first_day"

In [13]:
analysis_month

'2026-06-01'

In [14]:
og_df = _load_gtfs_data.load_schedule_rt_route_direction_summary(project_name=PROD_PROJECT,
        date_col=MONTH_DATE_COL,
        dataset_name=PROD_MART,
        start_date=last_year,
        end_date=analysis_month,
        file_name=file_name,)

Downloading: 100%|██████████|
query: SELECT * FROM  `cal-itp-data-infra`.`mart_gtfs_rollup`.`fct_monthly_schedule_rt_route_direction_summary` WHERE month_first_day >= DATE('2025-06-01') AND month_first_day <= DATE('2026-06-01')


In [15]:
og_df.sample()

,month,year,month_first_day,day_type,schedule_base64_url,schedule_name,tu_base64_url,tu_name,vp_base64_url,vp_name,route_name,direction_id,route_type,n_trips,daily_trips_all_day,n_route_ids,n_shapes,avg_stops_served,ttl_service_hours,ttl_flex_service_hours,daily_service_hours,daily_flex_service_hours,n_feeds,n_days,daily_trips_owl,daily_trips_early_am,daily_trips_am_peak,daily_trips_midday,daily_trips_pm_peak,daily_trips_evening,daily_trips_peak,daily_trips_offpeak,frequency_owl,frequency_early_am,frequency_am_peak,frequency_midday,frequency_pm_peak,frequency_evening,frequency_peak,frequency_offpeak,frequency_all_day,appeared_in_tu,appeared_in_vp,vp_num_distinct_updates,n_vp_trips,vp_extract_duration_minutes,vp_messages_per_minute,tu_num_distinct_updates,n_tu_trips,tu_extract_duration_minutes,tu_messages_per_minute,route_typology
124082,6,2026,2026-06-01,Saturday,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZWVkcz9vcGVyYXRvcl9pZD1TTQ==,Bay Area 511 SamTrans Schedule,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cGRhdGVzP2FnZW5jeT1TTQ==,Bay Area 511 SamTrans Trip Updates,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3ZlaGljbGVwb3NpdGlvbnM_YWdlbmN5PVNN,Bay Area 511 SamTrans VehiclePositions,281__281 Belle Haven Community Campus - Stanford,0,3,60,30.00,1,1.00,32.00,39.14,0.00,19.57,0.00,1,2,0.00,0.00,5.00,10.00,10.00,5.00,15.00,15.00,0.00,0.00,1.67,2.00,2.00,1.25,1.88,0.94,1.00,True,True,10132,59,3396,3.00,6684,59,2213,3.00,bus


In [39]:
m1 = pd.merge(og_df, crosswalk_df, left_on="schedule_name", right_on = "name", how="inner")

In [41]:
m1.sample()

,month,year,month_first_day,day_type,schedule_base64_url,schedule_name,tu_base64_url,tu_name,vp_base64_url,vp_name,route_name,direction_id,route_type,n_trips,daily_trips_all_day,n_route_ids,n_shapes,avg_stops_served,ttl_service_hours,ttl_flex_service_hours,daily_service_hours,daily_flex_service_hours,n_feeds,n_days,daily_trips_owl,daily_trips_early_am,daily_trips_am_peak,daily_trips_midday,daily_trips_pm_peak,daily_trips_evening,daily_trips_peak,daily_trips_offpeak,frequency_owl,frequency_early_am,frequency_am_peak,frequency_midday,frequency_pm_peak,frequency_evening,frequency_peak,frequency_offpeak,frequency_all_day,appeared_in_tu,appeared_in_vp,vp_num_distinct_updates,n_vp_trips,vp_extract_duration_minutes,vp_messages_per_minute,tu_num_distinct_updates,n_tu_trips,tu_extract_duration_minutes,tu_messages_per_minute,route_typology,name,analysis_name
92963,5,2026,2026-05-01,Sunday,aHR0cHM6Ly9naXRsYWIuY29tL0xBQ01UQS9ndGZzX2J1cy9yYXcvbWFzdGVyL2d0ZnNfYnVzLnppcA==,LA Metro Bus Schedule,aHR0cHM6Ly9hcGkuZ29zd2lmdC5seS9yZWFsLXRpbWUvbGFtZXRyby9ndGZzLXJ0LXRyaXAtdXBkYXRlcw==,LA Metro Bus Trip Updates,aHR0cHM6Ly9hcGkuZ29zd2lmdC5seS9yZWFsLXRpbWUvbGFtZXRyby9ndGZzLXJ0LXZlaGljbGUtcG9zaXRpb25z,LA Metro Bus Vehicle Positions,260__260/261 Metro Local Line,0,3,215,43.00,1,2.00,93.00,392.65,0.00,78.53,0.00,1,5,0.00,5.00,8.00,14.00,12.00,4.00,20.00,23.00,0.00,1.67,2.67,2.80,2.40,1.00,2.50,1.44,2.00,True,True,86859,207,29001,3.00,507970,215,171239,2.90,bus,LA Metro Bus Schedule,Los Angeles County Metropolitan Transportation Authority


In [43]:
m1.loc[m1.analysis_name.str.contains("Banning")].shape

(164, 54)